13. 罗马数字转整数

https://leetcode.cn/problems/roman-to-integer/description/?envType=study-plan-v2&envId=top-interview-150

本部分属于算法设计和字符串操作的范畴，即字符串处理问题。本题思路非常简单，先要构造字典查询，然后顺序扫描；因为存在 IV 这样的习惯问题，因此要考虑下一位比当前位大的情况，除此以外都是边界条件的细心考量，诸如
- n 长度为 1 的时候，循环失效应该直接返回；
- if i + 1 == n - 1 的情况单独处理。

In [3]:
class Solution:
    def romanToInt(self, s: str) -> int:
        n = len(s)
        roman = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
        i, answer = 0, 0
        if n == 1:
            return roman[s[0]]
        while i < n - 1:
            cur = roman[s[i]]
            nxt = roman[s[i + 1]]
            if cur >= nxt:
                answer += cur
                if i + 1 == n - 1:
                    answer += nxt
                    break
            else:
                answer += (nxt - cur)
                i += 1
                if i + 1 == n - 1:
                    answer += roman[s[i + 1]]
            i += 1
        return answer

solution = Solution()
solution.romanToInt("MCMXCIV")

1994

当然，如上写法是我自己写的，其中一个关键逻辑没有想明白，导致代码丑陋至极。可以说，代码的丑陋实际上某种程度上和代码的非最简性是相关的。也就是说，好的代码一定是外观优美的，所谓相由心生。这里，当 cur < nxt 时的处理非常不优雅：answer += (nxt - cur) 拆开来看，不就是此时减去 cur 吗？这种丑陋实际上不足以羞耻，因为其是人脑建模的一种反映，大脑在见到 IV 时，会将两个在视觉上靠近的字母“合并”处理，而不是真正的线性。

In [4]:
class Solution:
    def romanToInt(self, s: str) -> int:
        roman_map = {
            'I': 1, 'V': 5, 'X': 10, 'L': 50, 
            'C': 100, 'D': 500, 'M': 1000
        }
        total = 0
        
        for i in range(len(s) - 1):
            if roman_map[s[i]] < roman_map[s[i + 1]]:
                total -= roman_map[s[i]]
            else:
                total += roman_map[s[i]]
        
        total += roman_map[s[-1]]
        return total

12. 整数转罗马数字

https://leetcode.cn/problems/integer-to-roman/description/?envType=study-plan-v2&envId=top-interview-150

这道题做了半天做不出来，原因在于纠结很多小细节怎么实现。首先，看一下 AI 的做法，它先给了我一种最简单的实现方式。首先将特殊情况（即 4 和 9 开头的数位加入到编解码中）加入，然后直接贪心遍历。这里学到了一个语法即 `divmod(a, b)`，返回从 `a` 中最多含有 `b` 个数字的数量，和减去这些数字后的 number。

In [14]:
def int_to_roman(num: int) -> str:
    value_symbol_pairs = [
        (1000, "M"),
        (900, "CM"),
        (500, "D"),
        (400, "CD"),
        (100, "C"),
        (90, "XC"),
        (50, "L"),
        (40, "XL"),
        (10, "X"),
        (9, "IX"),
        (5, "V"),
        (4, "IV"),
        (1, "I"),
    ]
    
    result = []
    for value, symbol in value_symbol_pairs:
        if num == 0:
            break
        count, num = divmod(num, value)
        if count:
            result.append(symbol * count)
    return "".join(result)


print(int_to_roman(3))     # III
print(int_to_roman(4))     # IV
print(int_to_roman(9))     # IX
print(int_to_roman(58))    # LVIII
print(int_to_roman(1994))  # MCMXCIV

III
IV
IX
LVIII
MCMXCIV


受此启发，以下是我的代码。

In [ ]:
class Solution:
    def intToRoman(self, num: int) -> str:
        int2str = {
            1000: "M", 
            900: "CM", 
            500: "D", 
            400: "CD", 
            100: "C", 
            90: "XC", 
            50: "L", 
            40: "XL",
            10: "X", 
            9: "IX",
            5: "V", 
            4: "IV",
            1: "I"
        }
        answer = ""
        nums = [int(ch) * 10 ** (len(str(num)) - i - 1) for i, ch in enumerate(str(num))]
        for num in nums:
            for base in int2str.keys():
                if base > num:
                    continue
                elif base == num:
                    answer += int2str[base]
                    break
                else:
                    while num >= base:
                        num -= base
                        answer += int2str[base]
                    continue
        return answer

9. 回文数

https://leetcode.cn/problems/palindrome-number/description/?envType=study-plan-v2&envId=top-interview-150

这道题相对比较简单，用字符串处理最快。

In [ ]:
class Solution:
    def isPalindrome(self, x: int) -> bool:
        if x < 0:
            return False
        x = str(x)
        left, right = 0, len(x) - 1
        while left <= right:
            if x[left] != x[right]:
                return False
            left += 1
            right -= 1
        return True

14. 最长公共前缀

https://leetcode.cn/problems/longest-common-prefix/description/?envType=study-plan-v2&envId=top-interview-150

编写一个函数来查找字符串数组中的最长公共前缀。如果不存在公共前缀，返回空字符串 ""。

这道题并不难，首先要确定前缀题的做法。前缀的核心思想就是，拿定某一个前缀，然后用所有剩余的信息量（这里就是其他的字符串）去裁剪该前缀，这样确保复杂度控制在接近 $O(N)$，设所有字符串总长度为 $N$。

In [20]:
from typing import List

class Solution:
    def longestCommonPrefix(self, strs: List[str]) -> str:
        if len(strs) == 1:
            return strs[0]
        anchor = strs[0]
        for s in strs[1:]:
            if len(s) < len(anchor):
                anchor = anchor[:len(s)]
            i = len(anchor) - 1
            print(i, anchor[i], s[i])
            while anchor[i] != s[i]:
                anchor = anchor[:-1]
                i -= 1
                if anchor == "":
                    return anchor
        return anchor

当然，这道题在实现起来的时候，还是有很多坑的；首先是 anchor 和其他任何 string 如果有空值的时候，可以直接返回空；其次，有的时候不是慢慢从结尾开始不一样的，而是从中间开始不一样的，比如 `["cir", "car"]` 这个 case，如上的第一版写法就过不了，而只能 cover 诸如 `["flower","flow","flight"]` 这种缓和的情况。

不过，后来检查发现，一旦将后者的逻辑改进后，前者的逻辑被证明不是必须的，只是能更快跳出。

In [22]:
class Solution:
    def longestCommonPrefix(self, strs: List[str]) -> str:
        if len(strs) == 1:
            return strs[0]
        anchor = strs[0]
        if anchor == "":
            return ""
        for s in strs[1:]:
            if s == "":
                return ""
            if len(s) < len(anchor):
                anchor = anchor[:len(s)]
            i = len(anchor) - 1
            while i >= 0:
                if anchor[i] != s[i]:
                    anchor = anchor[:i]
                i -= 1
                if anchor == "":
                    return anchor
        return anchor

In [23]:
solution = Solution()
solution.longestCommonPrefix(["flower","flow","flight"])

'fl'

如上是我最后通过的正确写法，整体来看非常丑陋。如下是 AI 优化过的写法，核心思想没有变化但是更简洁，主要在于其将一个循环用了内置函数 `startswith`，这是值得学习的。

In [ ]:
class Solution:
    def longestCommonPrefix(self, strs: List[str]) -> str:
        if not strs:
            return ""
        prefix = strs[0]
        for s in strs[1:]:
            while not s.startswith(prefix):
                prefix = prefix[:-1]
                if not prefix:
                    return ""
        return prefix